# Semantic Segmentation of CUB-200 with U-Net

**Project Overview:**
This notebook implements a deep learning pipeline to solve a binary semantic segmentation task: extracting birds from the background in the CUB-200 dataset.

**Engineering Approach:**
1.  **Architecture:** We utilize a U-Net architecture. Instead of a standard contracting path, we use a pre-trained ResNet50 as the encoder to leverage high-level feature extraction capabilities.
2.  **Data Pipeline:** We implement a robust augmentation strategy using Albumentations to handle the high variance in bird poses and lighting.
3.  **Optimization:** We train using Binary Cross Entropy with Logits BCEWithLogitsLoss and monitor Intersection over Union, IoU, as our primary metric.

**Structure:**
*   `src.dataset`: Handles data loading and Albumentations.
*   `src.model`: Contains the custom U-Net with ResNet50 backbone.

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader

sys.path.append('../src')

from dataset import BirdsDataset
from model import UnetResNet50
from utils import seed_everything, calculate_iou

CONFIG = {
    "DATA_DIR": "../data",
    "WEIGHTS_DIR": "../weights",
    "IMAGE_SIZE": 256,
    "BATCH_SIZE": 16,
    "LR": 3e-4,
    "EPOCHS": 15,
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "SEED": 42
}

seed_everything(CONFIG["SEED"])
os.makedirs(CONFIG["WEIGHTS_DIR"], exist_ok=True)

## 1. Data Visualization & Augmentation

We'll use Albumentations for robust data augmentation.
- **Training:** Includes flips, shifts, rotations, coarse dropout, and color jitter.
- **Validation:** Only resizing and normalization.

In [ ]:
train_dataset = BirdsDataset(
    root_dir=CONFIG["DATA_DIR"],
    split="train",
    image_size=CONFIG["IMAGE_SIZE"]
)
val_dataset = BirdsDataset(
    root_dir=CONFIG["DATA_DIR"],
    split="val",
    image_size=CONFIG["IMAGE_SIZE"]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=False,
    num_workers=4
)

print(f"Training Samples: {len(train_dataset)}")
print(f"Validation Samples: {len(val_dataset)}")

In [ ]:
def denormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.permute(1, 2, 0).numpy()
    img = std * img + mean
    return np.clip(img, 0, 1)

def show_samples(dataset, n=3, title="Augmented Samples"):
    indices = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(n, 2, figsize=(8, 4*n))

    for i, idx in enumerate(indices):
        img, mask = dataset[idx]

        axes[i, 0].imshow(denormalize(img))
        axes[i, 0].set_title(f"Image {idx}")
        axes[i, 0].axis('off')

        axes[i, 1].imshow(mask.squeeze(), cmap='gray')
        axes[i, 1].set_title("Ground Truth Mask")
        axes[i, 1].axis('off')

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_samples(train_dataset, n=3, title="Training Data (Augmented)")

## 2. Model Architecture: U-Net with ResNet50

We'll initialize the custom U-Net model.
*   **Encoder:** We use a pre-trained ResNet50. This allows the model to start with a strong understanding of textures and edges.
*   **Decoder:** We use custom decoder blocks that upsample the features and concatenate them with the high-resolution features from the encoder.

In [ ]:
model = UnetResNet50(n_classes=1).to(CONFIG["DEVICE"])

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model Architecture: U-Net with ResNet50 Encoder")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

dummy = torch.randn(2, 3, 256, 256).to(CONFIG["DEVICE"])
with torch.no_grad():
    out = model(dummy)
print(f"Input Shape: {dummy.shape} -> Output Shape: {out.shape}")

## 3. Inference & Visualization

We will load the best weights trained using `src/train.py` and visualize the predictions on the Validation set.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG["LR"])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

def run_epoch(loader, is_train=True):
    model.train() if is_train else model.eval()

    epoch_loss = 0.0
    epoch_iou = 0.0

    pbar = tqdm(loader, desc="Train" if is_train else "Val", leave=False)

    for images, masks in pbar:
        images, masks = images.to(CONFIG["DEVICE"]), masks.to(CONFIG["DEVICE"])

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        batch_size = images.size(0)
        epoch_loss += loss.item() * batch_size

        with torch.no_grad():
            epoch_iou += calculate_iou(logits, masks) * batch_size

        pbar.set_postfix({"loss": loss.item()})

    return epoch_loss / len(loader.dataset), epoch_iou / len(loader.dataset)

In [ ]:
TRAIN_MODEL = True

history = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": []}
best_iou = 0.0

if TRAIN_MODEL:
    print("Starting Training...")
    for epoch in range(CONFIG["EPOCHS"]):
        t_loss, t_iou = run_epoch(train_loader, is_train=True)
        v_loss, v_iou = run_epoch(val_loader, is_train=False)

        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["train_iou"].append(t_iou)
        history["val_iou"].append(v_iou)

        scheduler.step(v_iou)

        print(f"Epoch {epoch+1}/{CONFIG['EPOCHS']} | "
              f"Train Loss: {t_loss:.4f} IoU: {t_iou:.4f} | "
              f"Val Loss: {v_loss:.4f} IoU: {v_iou:.4f}")

        if v_iou > best_iou:
            best_iou = v_iou
            torch.save(model.state_dict(), f"{CONFIG['WEIGHTS_DIR']}/best_unet.pth")
            print(f"--> Saved Best Model (IoU: {best_iou:.4f})")
else:
    print("Skipping training. Loading pre-trained weights.")
    if os.path.exists(f"{CONFIG['WEIGHTS_DIR']}/best_unet.pth"):
        model.load_state_dict(torch.load(f"{CONFIG['WEIGHTS_DIR']}/best_unet.pth"))
        print("Weights loaded.")
    else:
        print("No weights found! Please train the model.")

In [ ]:
if TRAIN_MODEL:
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title("Loss Curves")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_iou'], label='Train IoU')
    plt.plot(history['val_iou'], label='Val IoU')
    plt.title("IoU Curves")
    plt.legend()

    plt.show()

## 4. Evaluation & Qualitative Analysis

Now that the model is trained, let's visualize the results on the validation set.
We calculate the IoU for the entire validation set and then display prediction masks side-by-side with ground truth.

In [ ]:
model.load_state_dict(torch.load(f"{CONFIG['WEIGHTS_DIR']}/best_unet.pth"))
model.eval()

def visualize_prediction(index=None):
    if index is None:
        index = np.random.randint(0, len(val_dataset))

    img, mask = val_dataset[index]
    img_tensor = img.unsqueeze(0).to(CONFIG["DEVICE"])

    with torch.no_grad():
        logits = model(img_tensor)
        probs = torch.sigmoid(logits)
        pred_mask = (probs > 0.5).float().cpu().squeeze()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(denormalize(img))
    axes[0].set_title(f"Input Image (Idx {index})")

    axes[1].imshow(mask.squeeze(), cmap='gray')
    axes[1].set_title("Ground Truth")

    axes[2].imshow(pred_mask, cmap='gray')
    axes[2].set_title(f"Prediction")

    plt.show()

for _ in range(3):
    visualize_prediction()

## 5. Failure Case Analysis

To better understand model limitations, we specifically look for images where the model performed poorly $-$ low IoU.

In [ ]:
worst_samples = []

print("Scanning validation set for failure cases...")
for i in tqdm(range(len(val_dataset)), leave=False):
    img, mask = val_dataset[i]
    img_tensor = img.unsqueeze(0).to(CONFIG["DEVICE"])
    mask_tensor = mask.unsqueeze(0).to(CONFIG["DEVICE"])

    with torch.no_grad():
        logits = model(img_tensor)
        iou = calculate_iou(logits, mask_tensor)

    if iou < 0.5:
        worst_samples.append((i, iou))

worst_samples.sort(key=lambda x: x[1])

print(f"Found {len(worst_samples)} samples with IoU < 0.5")

for idx, iou in worst_samples[:3]:
    print(f"Sample {idx} - IoU: {iou:.4f}")
    visualize_prediction(idx)